In [9]:
from dotenv import load_dotenv
import os
from pathlib import Path
load_dotenv()
PATH_FINTOC_2022 = Path(os.getenv('PATH_FINTOC_2022'))

In [10]:
FINTOC_EN = PATH_FINTOC_2022/'data'/'en'
FINTOC_EN_PDF =FINTOC_EN/'pdf'
FINTOC_EN_ANNOTS = FINTOC_EN/'annots'
FINTOC_EN_PRED = FINTOC_EN/'preds'

JSON_EXTENSION = "fintoc4.json"

if not FINTOC_EN_PRED.exists():
    FINTOC_EN_PRED.mkdir()
FILE_PATHS = list(FINTOC_EN_PDF.iterdir())
FILE_PATHS.sort()
FILE_PATHS = FILE_PATHS[:10]

In [11]:
import pager
from pager.pager_pipe_line import pdf2region_row

from pager.doc_model import MinerPDFModel
from pager.doc_model import LogicTreeModel
from pager.doc_model.converters import PDF2LogicTree

from typing import Dict
import json

pdf_reader = MinerPDFModel({"page_model": pdf2region_row})
pdf2tree = PDF2LogicTree()
tree = LogicTreeModel()

In [12]:
file_path = FILE_PATHS[3]
pdf_reader.read_from_file(file_path)
pdf_reader.extract()
pdf2tree.convert(pdf_reader, tree)


doc_tree = tree.to_dict()

gt_path = FINTOC_EN_ANNOTS/(file_path.name+'.'+JSON_EXTENSION)
with open(gt_path, 'r') as f:
    data = json.load(f)

/home/daniil/project/PageR/env/lib/python3.12/site-packages/torch_geometric/nn/conv/tag_conv.py:102: UserWarning: Converting sparse tensor to CSR format for more efficient processing. Consider converting your sparse tensor to CSR format beforehand to avoid repeated conversion (got 'torch.sparse_coo')
  return spmm(adj_t, x, reduce=self.aggr)
/home/daniil/project/PageR/env/lib/python3.12/site-packages/torch/nn/modules/module.py:1775: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


In [13]:
from pager import PDF2Img, ImageModel
import matplotlib.pyplot as plt
from ipywidgets import interactive
from ipywidgets.widgets import BoundedIntText

pdf_model = pdf_reader.page_model.page_units[0].sub_model
pdf_model.path=file_path
region_model = pdf_reader.page_model.page_units[2].sub_model
img_model = ImageModel()
pdf2img = PDF2Img()

In [14]:
from pager.doc_model import PDF2LogicTree, LogicTreeModel
from pager.doc_model.dtype import Document, Page
from pager import Region
class CustomJson2LogicTree(PDF2LogicTree):
    def upload(self, reg_page, output_model:LogicTreeModel):
        reg_page = [(n, reg) for n, reg in  reg_page.items()]
        reg_page.sort(key= lambda x: x[0])
        reg_page = [regs for n, regs in reg_page]
        document = Document([Page(regions=regs, num_page=num) for num, regs in enumerate(reg_page)])
        regions = [reg for page in document.pages for reg in page.regions]
        tree = self.get_tree_from_doc(regions)
        self.set_level(regions, tree['parent'])
        
        output_model.document = document
        output_model.regions = regions
        output_model.edges = tree

class ExtractTree:
    def __init__(self):
        self.creater = CustomJson2LogicTree()
        self._font_from_doc = {}
        self._list_font_type = []

        
    def extract(self, tree:LogicTreeModel):
        doc_regions = [region for page in tree.document.pages for region in page.regions]
        region_num_page = [page.num_page for page in tree.document.pages for _ in page.regions]
        clusters = self.get_cluster(doc_regions)
        level_header_cluster = self.get_level_header(clusters)
        page_regions = self.get_regions(doc_regions, region_num_page, level_header_cluster)
        self.creater.upload(page_regions, tree)


    def get_regions(self, doc_regions, regions_num_page, level_header_cluster):
        num_cluster = [self._font_from_doc[vec] for vec in self._list_font_type]

        def seg_region(region, cl, num_page):
            
            if cl in level_header_cluster:
                region.header_level = level_header_cluster[cl]
                region.label = 'header'
            else:
                region.label = 'other'
            return region
        
        page_regions = dict()
        for region, cl, num_page in zip(doc_regions, num_cluster, regions_num_page):
            seg_region(region, cl, num_page)
            if num_page in page_regions:
                page_regions[num_page].append(region)
            else:
                page_regions[num_page] = [region]

        return page_regions

        
    def get_level_header(self, clusters):
        main_clust, main_font = self.get_main_cluster(clusters)
        header_cluster = {ind: cl[0] for ind, cl in clusters.items() if cl[0].font > main_font}
        header_cluster_index = list(header_cluster.keys())
        header_cluster_index.sort(key=lambda ind: header_cluster[ind].font, reverse=True)        
        level_header_cluster = {cl:level+1 for level, cl in enumerate(header_cluster_index)}
        return level_header_cluster
        
    def get_main_cluster(self, clusters):
        def get_words(reg):
            words = []
            for row in reg.rows:
                words.extend(row.words)
            return words
        max_clust = -1
        max_size = 0
        for ind, regions in clusters.items():
            
            count_word = len([w for reg in regions for w in get_words(reg)])
            if max_size < count_word:
                max_clust = ind
                max_size = count_word
        font_text =  clusters[max_clust][0].rows[0].font
        return max_clust, font_text

    def get_cluster(self, doc_regions):
        self._font_from_doc = {}
        self._list_font_type = []
        def get_vec_feature(row):
            fd = row.font.to_dict()
            name = fd['name']
            size = int(round(fd['size']/2))
            width = int(round(fd['width']/0.1))
            italic = int(round(fd['italic']/0.1))
            return (name, size, width, italic)
        
        def get_id_exist_cluster(feature):
            self._list_font_type.append(feature)
            if feature in self._font_from_doc.keys():
                 return self._font_from_doc[feature] 
            return None
                
        def add_new_cluster(feature):
            indexs= list(self._font_from_doc.values())
            index = max(indexs)+1 if len(indexs) != 0 else 0
            self._font_from_doc[feature] = index
            clusters[index] = []
            return index
            
        clusters = {}
        for region in doc_regions:
            feature_row = get_vec_feature(region)
            index = get_id_exist_cluster(feature_row)
            if index is None:
                index = add_new_cluster(feature_row)
            clusters[index].append(region)
        return clusters
        
       
     
   

extract_tree=ExtractTree() 

In [15]:
extract_tree.extract(tree)
num_page = 1
def plot_index(num_page):
    plt.figure(dpi=200)
    true_data = [d for d in data if d['page'] == num_page+1]
    for td in true_data:
        print(f"({td['depth']}) {td['text']}")
    plt.figure(dpi=200)
    pdf_model.num_page = num_page
    pdf2img.convert(pdf_model, img_model)
    
    img_model.show()
    page = tree.document.pages[num_page]
    for reg in page.regions:
        if reg.label == 'header':
            reg.segment.plot(text=f'({reg.header_level}) {reg.label} ({reg.metainfo['page']+1} p)' , text_size='xx-small', width=0.4)

        else:
            reg.segment.plot(color='k', width=0.3)
        print(reg.label, reg.font.to_dict(), reg.text[:10])
    for row in page.rows:
        row.segment.plot(width=0.2, color='r')


N = len(tree.document.pages)
interactive(plot_index, num_page=BoundedIntText(
    value=0,
    min=0,
    max=N-1,
    step=1,
    description='Index:',
    disabled=False
))

interactive(children=(BoundedIntText(value=0, description='Index:', max=78), Output()), _dom_classes=('widget-…

In [16]:
pdf_reader.pdf_json['document'], pdf_reader.pdf_json['pages'][0]

(PosixPath('/home/dataset/fintoc2022/data/en/pdf/BDLCM-Funds-Prospectus-VISA-28042020.pdf'),
 {'number': 0,
  'width': 596,
  'height': 843,
  'rows': [{'segment': {'x_top_left': 10,
     'y_top_left': 32,
     'width': 95,
     'height': 7},
    'text': 'VISA 2021/162143-8677-0-PC',
    'words': [{'segment': {'x_top_left': 10,
       'y_top_left': 32,
       'width': 16,
       'height': 7},
      'text': 'VISA',
      'font': {'fontname': 'Helvetica-Bold',
       'fontsize': 7.0,
       'is_normal': True}},
     {'segment': {'x_top_left': 28,
       'y_top_left': 32,
       'width': 77,
       'height': 7},
      'text': '2021/162143-8677-0-PC',
      'font': {'fontname': 'Helvetica-Bold',
       'fontsize': 7.0,
       'is_normal': True}}]},
   {'segment': {'x_top_left': 10, 'y_top_left': 40, 'width': 149, 'height': 7},
    'text': "L'apposition du visa ne peut en aucun cas servir",
    'words': [{'segment': {'x_top_left': 10,
       'y_top_left': 40,
       'width': 37,
       'hei